In [2]:
import numpy as np
import pandas as pd
import scipy
import anndata as ad
# import scanpy as sc

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
# print("Scanpy:", sc.__version__)
print("All imports successful!")

NumPy: 2.5.4
Pandas: 3.1.0
SciPy: 1.18.1
All imports successful!


In [3]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

In [5]:
download_base = Path('../../data/abc_atlas')
abc_cache = AbcProjectCache.from_cache_dir(download_base)

abc_cache.current_manifest

c:\Users\lilyg\miniforge3\envs\neuroscience490\Lib\site-packages\abc_atlas_access\abc_atlas_cache\cloud_cache.py:679: OutdatedManifestWarning: You are loading releases/20261031/manifest.json. A more up to date version of the dataset -- releases/20261130/manifest.json -- exists online. To see the changes between the two versions of the dataset, run
type.compare_manifests('releases/20261031/manifest.json', 'releases/20261130/manifest.json')
To load another version of the dataset, run
type.load_manifest('releases/20261130/manifest.json')
  warnings.warn(msg, OutdatedManifestWarning)


'releases/20261031/manifest.json'

In [6]:
abc_cache.list_metadata_files('WHB-10Xv3')

['anatomical_division_structure_map',
 'cell_metadata',
 'donor',
 'example_genes_all_cells_expression',
 'gene',
 'region_of_interest_structure_map']

In [7]:
abc_cache.list_metadata_files('WHB-taxonomy')

['cluster',
 'cluster_annotation_term',
 'cluster_annotation_term_set',
 'cluster_to_cluster_annotation_membership']

In [8]:
cell = abc_cache.get_metadata_dataframe(
    directory='WHB-10Xv3',
    file_name='cell_metadata',
    dtype={'cell_label': str}
)
cell.set_index('cell_label', inplace=True)
print("Number of cells = ", len(cell))
cell.head(5)
cell.columns

Number of cells =  3369219


C:\Users\lilyg\AppData\Local\Temp\ipykernel_36528\1535995828.py:6: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  cell.set_index('cell_label', inplace=True)


Index(['cell_barcode', 'barcoded_cell_sample_label', 'library_label',
       'feature_matrix_label', 'entity', 'brain_section_label',
       'library_method', 'donor_label', 'donor_sex', 'dataset_label', 'x', 'y',
       'cluster_alias', 'region_of_interest_label',
       'anatomical_division_label', 'abc_sample_id'],
      dtype='str')

In [9]:
region_counts = (
    cell.groupby("region_of_interest_label")
    .agg(
        n_cells=("donor_label", "size"),
        n_donors=("donor_label", "nunique"),
    )
    .sort_values(["n_donors", "n_cells"], ascending=False)
)

print(region_counts)

                          n_cells  n_donors
region_of_interest_label                   
Human M1C                  116576         4
Human MTG                  107301         3
Human CBV                   71874         3
Human SN                    59505         3
Human CA4C-DGC              56367         3
...                           ...       ...
Human HTHma                 10099         1
Human VA                     8573         1
Human CA3R                   8077         1
Human VPL                    6877         1
Human RN                     4714         1

[109 rows x 2 columns]


In [10]:
roi = abc_cache.get_metadata_dataframe(
    directory="WHB-10Xv3",
    file_name="region_of_interest_structure_map",
)

region_names = (
    roi.dropna(subset=["structure_name"])
    .groupby("region_of_interest_label")["structure_name"]
    .agg(lambda names: " / ".join(names.unique()))
)

region_summary = region_counts.join(region_names, validate="one_to_one")
display(region_summary)


,n_cells,n_donors,structure_name
region_of_interest_label,,,
Human M1C,116576,4,"primary motor cortex (area M1, area 4)"
Human MTG,107301,3,middle temporal gyrus
Human CBV,71874,3,cerebellar vermis
Human SN,59505,3,substantia nigra
Human CA4C-DGC,56367,3,caudal dentate gyrus / pyramidal cells of caud...
...,...,...,...
Human HTHma,10099,1,mammillary region of HTH
Human VA,8573,1,ventral anterior nucleus of thalamus
Human CA3R,8077,1,rostral CA3


In [11]:
cortical_labels = cell.loc[
    cell["anatomical_division_label"].eq("Cerebral cortex"),
    "region_of_interest_label",
].unique()

cortical_names = region_summary.loc[
    region_summary.index.isin(cortical_labels), "structure_name"
]

print("\n".join(cortical_names.dropna().unique()))

primary motor cortex (area M1, area 4)
middle temporal gyrus
medial (posterior) entorhinal cortex
subcentral cortex (gustatory cortex, area 43)
medial division of PPHC (area TH) / lateral division of PPHC (area TL)
superior temporal gyrus
dysgranular insular cortex
rostral division of IPC (area 40)
granular insular cortex
caudal portion of VFC (area 44) / rostral portion of VFC (area 45)
primary somatosensory cortex (area S1, areas 3,1,2)
subgenual (subcallosal) division of MFC (area 25)
temporal polar cortex (area TG, area 38)
piriform cortex
ventral division of PCC (area 23)
rostral division of SPC (area 5) / caudal division of SPC (area 7)
frontal agranular insular cortex (area FI)
caudal division of OFCi (area 13)
area 29 of retrosplenial cortex / area 30 of retrosplenial cortex
primary visual cortex (striate cortex, area V1/17)
primary auditory cortex (core)
lateral (anterior) entorhinal cortex
parastriate cortex (area V2, area 18)
ventral division of MFC (area 24)
caudal division

In [12]:
cortical_names = region_summary.loc[
    region_summary.index.isin(cortical_labels), "structure_name"
]

for label, name in cortical_names.dropna().items():
    print(f"{label} — {name}")

Human M1C — primary motor cortex (area M1, area 4)
Human MTG — middle temporal gyrus
Human MEC — medial (posterior) entorhinal cortex
Human A43 — subcentral cortex (gustatory cortex, area 43)
Human TH-TL — medial division of PPHC (area TH) / lateral division of PPHC (area TL)
Human STG — superior temporal gyrus
Human Idg — dysgranular insular cortex
Human A40 — rostral division of IPC (area 40)
Human Ig — granular insular cortex
Human A44-A45 — caudal portion of VFC (area 44) / rostral portion of VFC (area 45)
Human S1C — primary somatosensory cortex (area S1, areas 3,1,2)
Human A25 — subgenual (subcallosal) division of MFC (area 25)
Human A38 — temporal polar cortex (area TG, area 38)
Human Pir — piriform cortex
Human A23 — ventral division of PCC (area 23)
Human A5-A7 — rostral division of SPC (area 5) / caudal division of SPC (area 7)
Human FI — frontal agranular insular cortex (area FI)
Human A13 — caudal division of OFCi (area 13)
Human A29-A30 — area 29 of retrosplenial cortex / 

In [13]:
region_names = {
    "Human M1C": "M1",
    "Human V1C": "V1",
    "Human S1C": "S1",
}

human_isocortex = cell.loc[
    cell["region_of_interest_label"].isin(region_names)
    & cell["feature_matrix_label"].eq("WHB-10Xv3-Neurons"),
    ["region_of_interest_label", "donor_label", "cluster_alias"],
].copy()

human_isocortex["region"] = human_isocortex[
    "region_of_interest_label"
].map(region_names)

display(
    human_isocortex.groupby(["region", "donor_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(region_names.values(), fill_value=0)
)

donor_label,H18.30.001,H18.30.002,H19.30.001,H19.30.002
region,,,,
M1,48090,38757,11408,12001
V1,0,9258,12563,8791
S1,0,12886,11253,11323


In [14]:
membership = abc_cache.get_metadata_dataframe(
    directory="WHB-taxonomy",
    file_name="cluster_to_cluster_annotation_membership",
)

annotations = membership.pivot(
    index="cluster_alias",
    columns="cluster_annotation_term_set_name",
    values="cluster_annotation_term_name",
)[["supercluster", "neurotransmitter"]]

region = "PASTE_EXACT_REGION_LABEL_HERE"
human_pilot = cell.loc[
    cell["region_of_interest_label"].eq(region),
    ["donor_label", "cluster_alias"],
].join(annotations, on="cluster_alias", validate="many_to_one")

display(
    human_pilot.groupby(["neurotransmitter", "donor_label"], dropna=False)
    .size()
    .unstack("donor_label", fill_value=0)
)

donor_label
neurotransmitter


In [15]:
human_annotated = human_isocortex.join(
    annotations[["supercluster", "neurotransmitter"]],
    on="cluster_alias",
    validate="many_to_one",
)

display(
    human_annotated.groupby(
        ["region", "neurotransmitter", "donor_label"],
        dropna=False,
    )
    .size()
    .unstack("donor_label", fill_value=0)
)

donor_label                  H18.30.001  H18.30.002  H19.30.001  H19.30.002
region neurotransmitter                                                    
M1     GABA                       10837        8801        3399        3939
       GABA HDC                       1           0           0           0
       GABA VGLUT2                  108           0           0           0
       GABA VGLUT3                 2430        1305         569         584
       GLY                           10           0           0           0
       HDC VGLUT2                     7           0           0           0
       VGLUT1                     14595       10127        2817        2434
       VGLUT1 VGLUT2              19132       18202        4419        4791
       VGLUT1 VGLUT2 VGLUT3         551           7           0           1
       VGLUT2                       103           0           1           0
       VGLUT2 VGLUT3                  5           0           0           0
       NaN                          311         315         203         252
S1     GABA                           0        3112        3480        3655
       GABA VGLUT2                    0           1           0           0
       GABA VGLUT3                    0         369         636         608
       VGLUT1                         0        3505        2897        2760
       VGLUT1 VGLUT2                  0        5780        4057        4054
       VGLUT1 VGLUT2 VGLUT3           0           2           1           0
       VGLUT2                         0           2           0           1
       NaN                            0         115         182         245
V1     GABA                           0        1634        2543        1451
       GABA VGLUT2                    0           1           0           1
       GABA VGLUT3                    0          72         276         131
       VGLUT1                         0        1541        2937        1805
       VGLUT1 VGLUT2                  0        5952        6747        5380
       VGLUT1 VGLUT2 VGLUT3           0           1           0           0
       VGLUT2                         0           0           2           1
       VGLUT3                         0           0           1           0
       NaN                            0          57          57          22

In [16]:
transmitter = human_annotated["neurotransmitter"]

# Match labels made up entirely of VGLUT terms.
vglut_only = transmitter.str.fullmatch(r"VGLUT[123](?: VGLUT[123])*", na=False)
gaba_only = transmitter.eq("GABA").fillna(False)

human_grouped = human_annotated.assign(neuron_group="Other annotation")
human_grouped.loc[transmitter.isna(), "neuron_group"] = "Missing annotation"
human_grouped.loc[vglut_only, "neuron_group"] = "VGLUT-only"
human_grouped.loc[gaba_only, "neuron_group"] = "GABA-only"

display(
    human_grouped.groupby(["region", "neuron_group", "donor_label"])
    .size()
    .unstack("donor_label", fill_value=0)
)

human_pilot = human_grouped.loc[vglut_only | gaba_only].copy()

donor_label                H18.30.001  H18.30.002  H19.30.001  H19.30.002
region neuron_group                                                      
M1     GABA-only                10837        8801        3399        3939
       Missing annotation         311         315         203         252
       Other annotation          2556        1305         569         584
       VGLUT-only               34386       28336        7237        7226
S1     GABA-only                    0        3112        3480        3655
       Missing annotation           0         115         182         245
       Other annotation             0         370         636         608
       VGLUT-only                   0        9289        6955        6815
V1     GABA-only                    0        1634        2543        1451
       Missing annotation           0          57          57          22
       Other annotation             0          73         276         132
       VGLUT-only                   0        7494        9687        7186

In [17]:
from pathlib import Path

export_dir = Path("../data/region_exports")
export_dir.mkdir(exist_ok=True)

for filename, table in {
    "human_M1_S1_V1_metadata.csv": human_annotated
}.items():
    path = export_dir / filename
    table.to_csv(path, index_label="cell_label")
    print(f"{path.resolve()} | {len(table):,} rows | {path.stat().st_size / 1e6:.1f} MB")

C:\Users\lilyg\OneDrive\UPS\Fall 2026 UPS\NRSC 490\human-mouse-cortex\data\region_exports\human_M1_S1_V1_metadata.csv | 176,330 rows | 16.2 MB
